# BirdCLEF+ 2026 - Submission Notebook

**Environment:** Kaggle CPU notebook (no GPU, no internet, 90-min limit)

**Required Kaggle Datasets to attach:**
1. `birdclef-2026` - competition data
2. `birdclef-baseline-model-v5` - trained SED model .pth file
3. `birdclef-src` - source code (src/ and config/ directories)

In [ ]:
import os
import sys
import time

import numpy as np
import pandas as pd
import torch
import librosa
from tqdm.auto import tqdm

START_TIME = time.time()

# Detect environment
ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    KAGGLE_USER = "stochastix"
    DATA_DIR = "/kaggle/input/competitions/birdclef-2026"
    MODEL_PATH = f"/kaggle/input/datasets/{KAGGLE_USER}/birdclef-baseline-model-v5/best_model.pth"
    SRC_DIR = f"/kaggle/input/datasets/{KAGGLE_USER}/birdclef-src"
    sys.path.insert(0, SRC_DIR)
else:
    DATA_DIR = "../data"
    MODEL_PATH = "../models/best_model.pth"
    sys.path.insert(0, "..")

TEST_DIR = os.path.join(DATA_DIR, "test_soundscapes")
TAXONOMY_CSV = os.path.join(DATA_DIR, "taxonomy.csv")
SAMPLE_SUB = os.path.join(DATA_DIR, "sample_submission.csv")

print(f"ON_KAGGLE: {ON_KAGGLE}")
print(f"TEST_DIR: {TEST_DIR}")
print(f"MODEL_PATH: {MODEL_PATH}")
print(f"Model file exists: {os.path.exists(MODEL_PATH)}")

In [ ]:
import yaml
from src.audio import load_audio, audio_to_melspec, normalize_melspec
from src.transforms import spectrogram_to_tensor
from src.dataset import load_taxonomy
from src.model import get_model

# Load config
config_path = os.path.join(SRC_DIR, "config", "default.yaml") if ON_KAGGLE else "../config/default.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

audio_config = config["audio"]
SR = audio_config["sample_rate"]
DURATION = audio_config["duration"]
IN_CHANNELS = config["model"].get("in_channels", 1)
BATCH_SIZE = config["inference"].get("batch_size", 16)

# Load species list
species_list = load_taxonomy(TAXONOMY_CSV)
print(f"Species: {len(species_list)}")

## Load Model

In [ ]:
device = torch.device("cpu")

# Disable pretrained download — we load our own weights from best_model.pth
config["model"]["pretrained"] = False

model = get_model(config["model"])
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()

params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {params:,} parameters on {device}")
print(f"Time elapsed: {time.time() - START_TIME:.1f}s")

## Enumerate Test Windows

In [ ]:
import soundfile as sf

test_files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith(".ogg")])
print(f"Test soundscapes: {len(test_files)}")

# Build list of all windows
windows = []
for filename in test_files:
    file_path = os.path.join(TEST_DIR, filename)
    info = sf.info(file_path)
    total_dur = info.duration
    basename = os.path.splitext(filename)[0]

    start = 0.0
    while start + DURATION <= total_dur + 0.01:
        end = start + DURATION
        row_id = f"{basename}_{int(end)}"
        windows.append({
            "file_path": file_path,
            "start": start,
            "row_id": row_id,
        })
        start += DURATION

print(f"Total windows: {len(windows)}")
print(f"Time elapsed: {time.time() - START_TIME:.1f}s")

## Run Inference with TTA (Test-Time Augmentation)

TTA creates multiple versions of each audio window (original + augmented), runs all through the model in one batch, and averages the predictions for more robust results.

In [ ]:
NUM_TTA = 3  # original + 2 augmentations (keep low for 90-min CPU budget)

def make_tta_spectrograms(audio, sr, audio_config, in_channels):
    """Create original + augmented spectrograms from a single audio waveform.
    
    All operations are fast numpy array manipulations — no extra I/O.
    Returns a stacked tensor of shape (NUM_TTA, in_channels, n_mels, time_frames).
    """
    specs = []
    
    # 1. Original
    mel = audio_to_melspec(audio, sr, audio_config)
    mel = normalize_melspec(mel)
    specs.append(spectrogram_to_tensor(mel, in_channels))
    
    # 2. Time-shifted: shift audio by +0.5s (wrapping around)
    shift = int(0.5 * sr)  # 0.5 seconds
    shifted = np.roll(audio, shift)
    mel_shift = audio_to_melspec(shifted, sr, audio_config)
    mel_shift = normalize_melspec(mel_shift)
    specs.append(spectrogram_to_tensor(mel_shift, in_channels))
    
    # 3. Horizontal flip of spectrogram (time reversal)
    mel_flip = normalize_melspec(audio_to_melspec(audio[::-1].copy(), sr, audio_config))
    specs.append(spectrogram_to_tensor(mel_flip, in_channels))
    
    return torch.stack(specs)  # (NUM_TTA, C, H, W)


all_predictions = []

# Process windows in batches — each window produces NUM_TTA spectrograms
# We fill batches with TTA variants from multiple windows for efficient GPU/CPU use
TTA_BATCH_SIZE = BATCH_SIZE * NUM_TTA  # process BATCH_SIZE windows worth of TTA at once

for batch_start in tqdm(range(0, len(windows), BATCH_SIZE), desc="Inference (TTA)"):
    batch_windows = windows[batch_start:batch_start + BATCH_SIZE]
    batch_tta_tensors = []  # will hold NUM_TTA tensors per window
    batch_row_ids = []

    for w in batch_windows:
        # Load audio ONCE per window
        audio = load_audio(w["file_path"], sr=SR, duration=DURATION, offset=w["start"])
        # Create all TTA variants from same audio (fast numpy ops)
        tta_specs = make_tta_spectrograms(audio, SR, audio_config, IN_CHANNELS)
        batch_tta_tensors.append(tta_specs)
        batch_row_ids.append(w["row_id"])

    # Stack all TTA variants into one big batch: (num_windows * NUM_TTA, C, H, W)
    all_specs = torch.cat(batch_tta_tensors, dim=0)
    
    with torch.no_grad():
        output = model(all_specs)
        # Handle SED model dict output vs simple model tensor output
        logits = output["clipwise_logits"] if isinstance(output, dict) else output
        probs = torch.sigmoid(logits).numpy()  # (num_windows * NUM_TTA, 234)

    # Reshape to (num_windows, NUM_TTA, 234) and average across TTA axis
    num_windows = len(batch_windows)
    probs = probs.reshape(num_windows, NUM_TTA, -1)
    avg_probs = probs.mean(axis=1)  # (num_windows, 234)

    for row_id, prob_vec in zip(batch_row_ids, avg_probs):
        all_predictions.append((row_id, prob_vec))

print(f"Predictions: {len(all_predictions)} (from {len(all_predictions) * NUM_TTA} TTA forward passes)")
print(f"Time elapsed: {time.time() - START_TIME:.1f}s")

## Create Submission

In [ ]:
# Build submission DataFrame
if len(all_predictions) > 0:
    rows = []
    for row_id, probs in all_predictions:
        row = {"row_id": row_id}
        for species, prob in zip(species_list, probs):
            row[species] = float(prob)
        rows.append(row)
    submission = pd.DataFrame(rows)

    # Ensure column order matches sample submission
    if os.path.exists(SAMPLE_SUB):
        sample = pd.read_csv(SAMPLE_SUB, nrows=1)
        submission = submission[sample.columns]
else:
    # No test files during Save & Run All — use sample submission as placeholder
    print("No test soundscapes found (expected during Save & Run All)")
    submission = pd.read_csv(SAMPLE_SUB)

# Save
submission.to_csv("submission.csv", index=False)
print(f"Submission shape: {submission.shape}")
print(f"Columns: {len(submission.columns)} (1 row_id + {len(submission.columns) - 1} species)")
submission.head()

## Sanity Checks

In [ ]:
# Verify submission
assert submission.shape[1] == len(species_list) + 1, f"Expected {len(species_list) + 1} columns, got {submission.shape[1]}"
assert not submission.isnull().any().any(), "Submission contains NaN values!"

if len(all_predictions) > 0:
    prob_cols = submission.columns[1:]
    prob_values = submission[prob_cols].values
    assert prob_values.min() >= 0, f"Min probability is {prob_values.min()}"
    assert prob_values.max() <= 1, f"Max probability is {prob_values.max()}"
    print(f"Probability range: [{prob_values.min():.6f}, {prob_values.max():.6f}]")
    print(f"Mean probability: {prob_values.mean():.6f}")

print("All checks passed!")
print(f"Rows: {len(submission)}")
print(f"\nTotal runtime: {time.time() - START_TIME:.1f}s ({(time.time() - START_TIME)/60:.1f} min)")